# 03 - Construccion OBT

Objetivo: construir `ANALYTICS.OBT_TRIPS` (grano 1 viaje) con columnas minimas, derivadas y metadatos de lineage segun el PDF.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [2]:
spark_snowflake_pkg = os.getenv('SPARK_SNOWFLAKE_PACKAGE', 'net.snowflake:spark-snowflake_2.12:3.1.8')
snowflake_jdbc_pkg = os.getenv('SNOWFLAKE_JDBC_PACKAGE', 'net.snowflake:snowflake-jdbc:3.14.4')

spark = (
    SparkSession.builder
    .appName('PSet3 Construccion OBT')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.jars.packages', f"{spark_snowflake_pkg},{snowflake_jdbc_pkg}")
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)

print(spark.version)

3.5.0


## 1) Configuracion

In [3]:
def get_env(name, default=None, required=False):
    value = os.getenv(name, default)
    if required and (value is None or str(value).strip() == ''):
        raise ValueError(f'Falta variable de entorno requerida: {name}')
    return value

snowflake_host = get_env('SNOWFLAKE_HOST', '').strip()
snowflake_account = get_env('SNOWFLAKE_ACCOUNT', '').strip()
snowflake_port = get_env('SNOWFLAKE_PORT', '443').strip()

if not snowflake_host:
    if not snowflake_account:
        raise ValueError('Debes configurar SNOWFLAKE_HOST o SNOWFLAKE_ACCOUNT')
    snowflake_host = f"{snowflake_account}.snowflakecomputing.com"

if snowflake_port and snowflake_port != '443':
    snowflake_host = f"{snowflake_host}:{snowflake_port}"

sf_options = {
    'sfURL': snowflake_host,
    'sfUser': get_env('SNOWFLAKE_USER', required=True),
    'sfPassword': get_env('SNOWFLAKE_PASSWORD', required=True),
    'sfRole': get_env('SNOWFLAKE_ROLE', required=True),
    'sfWarehouse': get_env('SNOWFLAKE_WAREHOUSE', required=True),
    'sfDatabase': get_env('SNOWFLAKE_DATABASE', required=True),
    'sfSchema': get_env('SNOWFLAKE_SCHEMA_RAW', 'RAW'),
}

raw_schema = get_env('SNOWFLAKE_SCHEMA_RAW', 'RAW')
analytics_schema = get_env('SNOWFLAKE_SCHEMA_ANALYTICS', 'ANALYTICS')
source_table = f"{raw_schema}.TRIPS_UNIFIED"
target_table = f"{analytics_schema}.OBT_TRIPS"

build_start_year = int(get_env('OBT_BUILD_START_YEAR', get_env('START_YEAR', '2015')))
build_end_year = int(get_env('OBT_BUILD_END_YEAR', get_env('END_YEAR', '2025')))
build_months = [m.strip() for m in get_env('OBT_BUILD_MONTHS', get_env('MONTHS', '01,02,03,04,05,06,07,08,09,10,11,12')).split(',')]

{k: ('***' if 'Password' in k else v) for k, v in sf_options.items()}

{'sfURL': 'DVNRQOK-CRC15844.snowflakecomputing.com',
 'sfUser': 'tonyfajardo1d',
 'sfPassword': '***',
 'sfRole': 'ACCOUNTADMIN',
 'sfWarehouse': 'PSET3_WH',
 'sfDatabase': 'DM_PSET3',
 'sfSchema': 'RAW'}

## 2) Leer tabla unificada fuente

In [4]:
df_src = (
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('dbtable', source_table)
    .load()
)

df_src = df_src.select([F.col(c).alias(c.lower()) for c in df_src.columns])

df_src = df_src.filter(
    (F.col('source_year').cast('int') >= F.lit(build_start_year)) &
    (F.col('source_year').cast('int') <= F.lit(build_end_year)) &
    (F.col('source_month').isin(*build_months))
)

print('rows source filtradas =', df_src.count())
df_src.select('service_type', 'source_year', 'source_month').show(10, truncate=False)

rows source filtradas = 866674985
+------------+-----------+------------+
|service_type|source_year|source_month|
+------------+-----------+------------+
|yellow      |2025       |06          |
|yellow      |2025       |06          |
|yellow      |2025       |06          |
|yellow      |2025       |06          |
|yellow      |2025       |06          |
|yellow      |2025       |06          |
|yellow      |2025       |06          |
|yellow      |2025       |06          |
|yellow      |2025       |06          |
|yellow      |2025       |06          |
+------------+-----------+------------+
only showing top 10 rows



## 3) Construir OBT con columnas minimas + derivadas

In [5]:
df_obt_pre = (
    df_src
    .withColumn('pickup_datetime', F.col('pickup_datetime').cast('timestamp'))
    .withColumn('dropoff_datetime', F.col('dropoff_datetime').cast('timestamp'))
    .withColumn('pickup_date', F.to_date('pickup_datetime'))
    .withColumn('pickup_hour', F.hour('pickup_datetime'))
    .withColumn('dropoff_date', F.to_date('dropoff_datetime'))
    .withColumn('dropoff_hour', F.hour('dropoff_datetime'))
    .withColumn('day_of_week', F.dayofweek('pickup_datetime'))
    .withColumn('month', F.month('pickup_datetime'))
    .withColumn('year', F.year('pickup_datetime'))
    .filter(F.col('pickup_datetime').isNotNull())
    .filter(F.col('dropoff_datetime').isNotNull())
    .filter(F.col('pickup_datetime') >= F.to_timestamp(F.lit(f'{build_start_year}-01-01 00:00:00')))
    .filter(F.col('pickup_datetime') < F.to_timestamp(F.lit(f'{build_end_year + 1}-01-01 00:00:00')))
    .filter(F.col('dropoff_datetime') >= F.to_timestamp(F.lit(f'{build_start_year}-01-01 00:00:00')))
    .filter(F.col('dropoff_datetime') < F.to_timestamp(F.lit(f'{build_end_year + 1}-01-01 00:00:00')))
    .filter(F.abs(F.col('year') - F.col('source_year').cast('int')) <= 1)
    .withColumn('trip_duration_min', (F.unix_timestamp('dropoff_datetime') - F.unix_timestamp('pickup_datetime')) / F.lit(60.0))
    .withColumn('avg_speed_mph', F.when(F.col('trip_duration_min') > 0, F.col('trip_distance') / (F.col('trip_duration_min') / F.lit(60.0))))
    .withColumn('tip_pct', F.when(F.col('fare_amount') > 0, F.col('tip_amount') / F.col('fare_amount')))
    .withColumn('source_service', F.col('service_type'))
    .select(
        # clave natural
        'trip_nk',

        # tiempo
        'pickup_datetime', 'dropoff_datetime', 'pickup_date', 'pickup_hour',
        'dropoff_date', 'dropoff_hour', 'day_of_week', 'month', 'year',

        # ubicacion
        'pu_location_id', 'pu_zone', 'pu_borough',
        'do_location_id', 'do_zone', 'do_borough',

        # servicio y codigos
        'service_type', 'vendor_id', 'vendor_name',
        'rate_code_id', 'rate_code_desc',
        'payment_type', 'payment_type_desc',
        'trip_type', 'trip_type_desc',

        # viaje
        'passenger_count', 'trip_distance', 'store_and_fwd_flag',

        # tarifas
        'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
        'improvement_surcharge', 'congestion_surcharge', 'airport_fee', 'total_amount',

        # derivadas
        'trip_duration_min', 'avg_speed_mph', 'tip_pct',

        # lineage
        'run_id', 'ingested_at_utc', 'source_service', 'source_year', 'source_month'
    )
)

# Calidad base obligatoria para OBT (alineado al PDF)
df_obt = (
    df_obt_pre
    .filter(F.col('trip_distance').isNotNull())
    .filter(F.col('trip_distance') >= 0)
    .filter(F.col('trip_distance') <= 100)
    .filter(F.col('trip_duration_min').isNotNull())
    .filter(F.col('trip_duration_min') >= 1)
    .filter(F.col('trip_duration_min') <= 240)
    .filter(F.col('fare_amount').isNull() | (F.col('fare_amount') >= 0))
    .filter(F.col('total_amount').isNull() | (F.col('total_amount') >= 0))
    .filter(F.col('avg_speed_mph').isNull() | ((F.col('avg_speed_mph') >= 1) & (F.col('avg_speed_mph') <= 120)))
    .filter(F.col('passenger_count').isNull() | ((F.col('passenger_count') >= 1) & (F.col('passenger_count') <= 6)))
)

rows_pre = df_obt_pre.count()
rows_post = df_obt.count()
rows_discarded = rows_pre - rows_post
discard_pct = round((rows_discarded / rows_pre) * 100, 4) if rows_pre else 0
print('rows_pre_quality =', rows_pre)
print('rows_post_quality =', rows_post)
print('rows_discarded =', rows_discarded)
print('discard_pct =', discard_pct, '%')

df_obt.printSchema()
df_obt.show(10, truncate=False)

rows_pre_quality = 866670730
rows_post_quality = 844201815
rows_discarded = 22468915
discard_pct = 2.5926 %
root
 |-- trip_nk: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- dropoff_date: date (nullable = true)
 |-- dropoff_hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- pu_location_id: decimal(38,0) (nullable = true)
 |-- pu_zone: string (nullable = true)
 |-- pu_borough: string (nullable = true)
 |-- do_location_id: decimal(38,0) (nullable = true)
 |-- do_zone: string (nullable = true)
 |-- do_borough: string (nullable = true)
 |-- service_type: string (nullable = false)
 |-- vendor_id: decimal(38,0) (nullable = true)
 |-- vendor_name: string (nullable = false)
 |-- rate_code_id: decimal(38,0) (nullable = true)


## 4) Escritura en ANALYTICS.OBT_TRIPS

In [6]:
(
    df_obt.write
    .format('snowflake')
    .options(**sf_options)
    .option('dbtable', target_table)
    .mode('overwrite')
    .save()
)

print(f'Escritura OK -> {target_table}')

Escritura OK -> ANALYTICS.OBT_TRIPS


## 5) Verificaciones

In [7]:
q_counts = f"""
SELECT service_type, source_year, source_month, COUNT(*) AS rows_obt
FROM {target_table}
GROUP BY 1,2,3
ORDER BY 1,2,3
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_counts)
    .load()
    .show(50, truncate=False)
)

+------------+-----------+------------+--------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|ROWS_OBT|
+------------+-----------+------------+--------+
|green       |2015       |01          |1469403 |
|green       |2015       |02          |1536080 |
|green       |2015       |03          |1677899 |
|green       |2015       |04          |1620980 |
|green       |2015       |05          |1741602 |
|green       |2015       |06          |1597723 |
|green       |2015       |07          |1503389 |
|green       |2015       |08          |1495565 |
|green       |2015       |09          |1451661 |
|green       |2015       |10          |1581613 |
|green       |2015       |11          |1481843 |
|green       |2015       |12          |1558693 |
|green       |2016       |01          |1398910 |
|green       |2016       |02          |1465607 |
|green       |2016       |03          |1528739 |
|green       |2016       |04          |1497030 |
|green       |2016       |05          |1489462 |
|green       |2016  

In [8]:
q_quality = f"""
SELECT
  SUM(CASE WHEN pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
  SUM(CASE WHEN dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,
  SUM(CASE WHEN trip_duration_min < 0 THEN 1 ELSE 0 END) AS negative_duration,
  SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_distance
FROM {target_table}
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_quality)
    .load()
    .show(truncate=False)
)

+--------------------+---------------------+-----------------+-----------------+
|NULL_PICKUP_DATETIME|NULL_DROPOFF_DATETIME|NEGATIVE_DURATION|NEGATIVE_DISTANCE|
+--------------------+---------------------+-----------------+-----------------+
|0                   |0                    |0                |0                |
+--------------------+---------------------+-----------------+-----------------+



In [9]:
# chequeo simple de idempotencia por rerun (mismo count tras overwrite)
q_total = f"SELECT COUNT(*) AS total_obt FROM {target_table}"
total_obt = (
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_total)
    .load()
    .collect()[0]['TOTAL_OBT']
)
print('total_obt =', total_obt)

total_obt = 844201815


## 6) Verificacion de idempotencia en RAW (reingesta de un mes)

Este bloque valida no-duplicacion por `trip_nk` para un particion mensual.

Si quieres evidencia completa del PDF, reejecuta `01_ingesta_parquet_raw.ipynb` para el mismo mes y vuelve a correr esta seccion (el resultado debe seguir en 0 duplicados).

In [18]:
idemp_service = get_env('IDEMP_SERVICE', 'yellow')
idemp_year = int(get_env('IDEMP_YEAR', '2024'))
idemp_month = get_env('IDEMP_MONTH', '12')

raw_schema = get_env('SNOWFLAKE_SCHEMA_RAW', 'RAW')
raw_table = f"{raw_schema}.TRIPS"

q_idemp = f"""
WITH part AS (
  SELECT trip_nk
  FROM {raw_table}
  WHERE service_type = '{idemp_service}'
    AND source_year = {idemp_year}
    AND source_month = '{idemp_month}'
), dups AS (
  SELECT trip_nk, COUNT(*) AS c
  FROM part
  GROUP BY 1
  HAVING COUNT(*) > 1
)
SELECT
  '{idemp_service}' AS service_type,
  {idemp_year} AS source_year,
  '{idemp_month}' AS source_month,
  (SELECT COUNT(*) FROM part) AS rows_partition,
  (SELECT COUNT(DISTINCT trip_nk) FROM part) AS distinct_trip_nk,
  (SELECT COUNT(*) FROM dups) AS duplicated_keys
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_idemp)
    .load()
    .show(truncate=False)
)

+------------+-----------+------------+--------------+----------------+---------------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|ROWS_PARTITION|DISTINCT_TRIP_NK|DUPLICATED_KEYS|
+------------+-----------+------------+--------------+----------------+---------------+
|yellow      |2024       |12          |3597775       |3597775         |0              |
+------------+-----------+------------+--------------+----------------+---------------+

